# 06 — Geocoding (passo 13: OpenStreetMap / Nominatim)

**Objetivo:** para quem ainda está sem coordenada, consultar o Nominatim com o endereço da Receita.

**Entradas:** `df_espacial` e `aux_estab` (05), `df_lotes` (03, para a área da cidade); `02_dados_editados/geocoding_nominatim.jsonl` (respostas já guardadas).

**Saídas:** `03_processamento/06_geocoding_nominatim/df_espacial` (após o passo 13). As respostas novas são acrescentadas ao `geocoding_nominatim.jsonl`, com backup antes em `02_dados_editados/_backup/`.

**Como rodar:** de cima para baixo. Cada rodada faz no máximo `parametros.geocoding.max_consultas` consultas (1 por segundo); as que faltarem ficam para a próxima. Depois de cada rodada: `python executar_tudo.py --ciclo` (06 a 09). Para não consultar o serviço, ponha `parametros.geocoding.ativo: false`.

**Seções:** 1. Passo 13 (geocoding) · 2. Gravação e validação · 3. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("06_geocoding_nominatim", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# Funções e constantes do projeto usadas pelas seções (src/enderecos.py, src/cascata.py e src/geo.py).
from src.enderecos import (
    normalizar_numero, normalizar_texto, nucleo_de_texto,
)
from src.cascata import (
    gravar_resultado, resumo_passo, selecionar_pendentes,
)

from src import cascata

# Limiares das funções da cascata (src/cascata.py), lidos do config.yaml.
cascata.configurar(cfg)

# ===== PARÂMETROS (config.yaml: parametros.geocoding) =====
GEOCODING_ATIVO = cfg.parametro("geocoding.ativo")                    # False = não consulta o serviço
GEOCODING_URL = cfg.parametro("geocoding.url")
GEOCODING_USER_AGENT = cfg.parametro("geocoding.user_agent")          # o Nominatim exige identificação e contato
GEOCODING_INTERVALO_S = cfg.parametro("geocoding.intervalo_s")        # segundos entre consultas
GEOCODING_MAX_CONSULTAS = cfg.parametro("geocoding.max_consultas")    # consultas novas por rodada
GEOCODING_TEMPO_LIMITE_S = cfg.parametro("geocoding.tempo_limite_s")
GEOCODING_CIDADE = cfg.parametro("geocoding.cidade")
GEOCODING_ESTADO = cfg.parametro("geocoding.estado")
GEOCODING_PAIS = cfg.parametro("geocoding.pais")
GEOCODING_NOTA_MINIMA_VIA = cfg.parametro("geocoding.nota_minima_via")
GEOCODING_MARGEM_GRAUS = cfg.parametro("geocoding.margem_graus")

# ===== ENTRADAS =====
df_espacial = fontes.ler_processamento(cfg, "espacial_05")    # 05
aux_estab = fontes.ler_processamento(cfg, "aux_estab")        # 05
df_lotes = fontes.ler_processamento(cfg, "df_lotes")          # 03
# Respostas do Nominatim já guardadas (arquivo mantido em 02_dados_editados; criado na primeira rodada).
ARQ_CACHE_GEOCODING = cfg.caminho("editados.geocoding_nominatim")
fontes.caminho_editado(cfg, "geocoding_nominatim", obrigatorio=False)

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Passo 13: geocoding (OpenStreetMap / Nominatim)
Para quem sobrou, consulta o serviço público Nominatim, do OpenStreetMap, com o endereço da Receita (número + via, cidade, estado e país), restrito à área da cidade. Uma consulta por endereço diferente, **no máximo uma por segundo**, como pede a política de uso do serviço.
Cada resposta é guardada em `02_dados_editados/geocoding_nominatim.jsonl` na hora em que chega (antes de cada rodada com consultas novas, o arquivo ganha uma cópia datada em `_backup/`): se o bloco parar no meio (ou a rede cair), a próxima rodada continua de onde parou e nunca repete uma consulta. Cada rodada faz no máximo `parametros.geocoding.max_consultas` (config.yaml) consultas novas; o log mostra o tempo previsto.
Um resultado só é aceito se estiver na cidade e o nome da rua for parecido com o da Receita (nota mínima 85). Confiança **baixa** quando o número foi encontrado, e **muito baixa** quando só a rua foi encontrada.
**Antes de rodar, confira o e-mail em `parametros.geocoding.user_agent` (config.yaml).** Sem rede ou com o serviço fora, o bloco avisa e segue, usando só as respostas já guardadas. Dados © colaboradores do OpenStreetMap (licença ODbL).

In [ ]:
# Bibliotecas para chamar o serviço de geocoding pela internet (já vêm com o Python).
import urllib.request
import urllib.parse
import urllib.error
# Biblioteca para comparar o nome da rua devolvida com o da Receita (nota de 0 a 100).
from rapidfuzz import fuzz

log.info("SEÇÃO 1: passo 13 (geocoding no OpenStreetMap / Nominatim)...")
inicio_bloco = time.time()
NOME_PASSO = "geocoding (OpenStreetMap / Nominatim)"
# O arquivo onde cada resposta do serviço é guardada (ARQ_CACHE_GEOCODING, em 02_dados_editados)
# vem da preparação. Uma linha por consulta: assim nada é perguntado duas vezes.

# ===================================================================
# ETAPA 1: SELECIONAR OS ESTABELECIMENTOS E MONTAR AS CONSULTAS
# ===================================================================
log.info("Etapa 1: selecionando os estabelecimentos e montando as consultas...")
pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 13, exigir_num=False, exigir_cep=False)
# Núcleo da via de cada estabelecimento (para conferir o nome da rua que o serviço devolver).
est["nucleo"] = aux_estab["nucleo"].reindex(est["idx"]).values
# Chave de cada consulta: via + número. Vários CNPJs no mesmo endereço viram uma consulta só.
est["chave"] = est["via_e"].astype(str) + "|" + est["num"].fillna("").astype(str)
consultas = est.drop_duplicates("chave")[["chave", "via_e", "num", "nucleo"]].reset_index(drop=True)
log.info(f"Etapa 1 concluída! {len(est):,} estabelecimentos | {len(consultas):,} endereços diferentes.")

# ===================================================================
# ETAPA 2: LER AS RESPOSTAS JÁ GUARDADAS
# ===================================================================
log.info("\nEtapa 2: lendo as respostas já guardadas no cache do geocoding...")
respostas = {}
if ARQ_CACHE_GEOCODING.is_file():
    with open(ARQ_CACHE_GEOCODING, encoding="utf-8") as arquivo_cache:
        for linha in arquivo_cache:
            linha = linha.strip()
            if not linha:
                continue
            try:
                registro = json.loads(linha)
                respostas[registro["chave"]] = registro["resposta"]
            except Exception:
                # Linha estragada (por exemplo, gravação interrompida): é ignorada e a consulta será refeita.
                continue
faltam = consultas[~consultas["chave"].isin(respostas)]
log.info(f"Etapa 2 concluída! Já respondidas: {len(consultas) - len(faltam):,} | faltam: {len(faltam):,}.")

# ===================================================================
# ETAPA 3: CONSULTAR O SERVIÇO (SÓ O QUE FALTA)
# ===================================================================
log.info("\nEtapa 3: consultando o serviço...")

# Caixa onde o resultado pode cair: a área dos lotes da cidade, com uma margem.
lat_min = df_lotes["lat"].min() - GEOCODING_MARGEM_GRAUS
lat_max = df_lotes["lat"].max() + GEOCODING_MARGEM_GRAUS
lon_min = df_lotes["lon"].min() - GEOCODING_MARGEM_GRAUS
lon_max = df_lotes["lon"].max() + GEOCODING_MARGEM_GRAUS

# Motivos para não consultar (o passo continua com as respostas que já estão no cache).
if not GEOCODING_ATIVO:
    motivo_pular = "parametros.geocoding.ativo = false no config.yaml"
elif "PREENCHA" in GEOCODING_USER_AGENT:
    motivo_pular = "falta colocar o seu e-mail em GEOCODING_USER_AGENT, no config.yaml (exigência do Nominatim)"
elif faltam.empty:
    motivo_pular = "nada a consultar"
else:
    motivo_pular = None

if motivo_pular:
    log.info(f"    Nenhuma consulta feita: {motivo_pular}.")
else:
    a_consultar = faltam.head(GEOCODING_MAX_CONSULTAS)
    tempo_previsto = len(a_consultar) * GEOCODING_INTERVALO_S
    log.info(f"    Serão feitas {len(a_consultar):,} consultas (tempo previsto: {formatar_tempo(tempo_previsto)}). "
          f"Ficam para a próxima rodada: {len(faltam) - len(a_consultar):,}.")
    # Garante que a pasta 02_dados_editados existe.
    ARQ_CACHE_GEOCODING.parent.mkdir(parents=True, exist_ok=True)
    # Antes de acrescentar respostas novas, guarda uma cópia datada do arquivo em 02_dados_editados/_backup.
    if ARQ_CACHE_GEOCODING.is_file():
        log.info("    Fazendo o backup do cache do geocoding antes de acrescentar respostas...")
        fontes.backup(cfg, ARQ_CACHE_GEOCODING)
        log.info("    Backup do cache do geocoding feito!")
    inicio_consultas = time.time()
    ultima_chamada = 0.0
    feitas = 0
    with open(ARQ_CACHE_GEOCODING, "a", encoding="utf-8") as arquivo_cache:
        for via, num, chave in zip(a_consultar["via_e"], a_consultar["num"], a_consultar["chave"]):
            # Endereço no formato que o Nominatim entende: "número via", cidade, estado e país separados.
            parametros = {
                "street": f"{num} {via}" if isinstance(num, str) else via,
                "city": GEOCODING_CIDADE, "state": GEOCODING_ESTADO, "country": GEOCODING_PAIS,
                "format": "jsonv2", "addressdetails": 1, "limit": 5, "countrycodes": "br",
                "viewbox": f"{lon_min},{lat_max},{lon_max},{lat_min}", "bounded": 1,
            }
            url = GEOCODING_URL + "?" + urllib.parse.urlencode(parametros)
            pedido = urllib.request.Request(url, headers={"User-Agent": GEOCODING_USER_AGENT,
                                                          "Accept-Language": "pt-BR"})
            resposta = None
            for tentativa in (1, 2):
                # Respeita o intervalo mínimo entre duas consultas.
                espera = GEOCODING_INTERVALO_S - (time.time() - ultima_chamada)
                if espera > 0:
                    time.sleep(espera)
                ultima_chamada = time.time()
                try:
                    with urllib.request.urlopen(pedido, timeout=GEOCODING_TEMPO_LIMITE_S) as r:
                        resposta = json.loads(r.read().decode("utf-8"))
                    break
                except urllib.error.HTTPError as erro:
                    # 429 = muitas consultas; 5xx = problema no servidor. Espera 1 minuto e tenta uma vez mais.
                    if tentativa == 1 and (erro.code == 429 or erro.code >= 500):
                        log.info(f"    O serviço respondeu {erro.code}: esperando 60 s para tentar de novo...")
                        time.sleep(60)
                        continue
                    log.info(f"    O serviço recusou a consulta (HTTP {erro.code}).")
                    break
                except Exception as erro:
                    # Sem internet, bloqueio da rede ou serviço fora do ar.
                    log.info(f"    Falha de conexão ({type(erro).__name__}).")
                    break
            if resposta is None:
                log.info("    As consultas foram interrompidas; as respostas já recebidas estão guardadas.")
                break
            # Guarda a resposta na hora (mesmo vazia), para nunca repetir a consulta.
            respostas[chave] = resposta
            arquivo_cache.write(json.dumps({"chave": chave, "data": time.strftime("%Y-%m-%d %H:%M:%S"),
                                            "resposta": resposta}, ensure_ascii=False) + "\n")
            arquivo_cache.flush()
            feitas += 1
            if feitas % 100 == 0:
                decorrido = time.time() - inicio_consultas
                restante = decorrido / feitas * (len(a_consultar) - feitas)
                log.info(f"    {feitas:,} de {len(a_consultar):,} consultas feitas (faltam ~{formatar_tempo(restante)})...")
    log.info(f"    Consultas feitas nesta rodada: {feitas:,}.")
log.info("Etapa 3 concluída!")

# ===================================================================
# ETAPA 4: AVALIAR AS RESPOSTAS
# ===================================================================
log.info("\nEtapa 4: avaliando as respostas (cidade, nome da rua e precisão)...")
CIDADE_NORMALIZADA = normalizar_texto(pd.Series([GEOCODING_CIDADE])).iloc[0]


def avaliar_resposta(resposta, num, nucleo):
    """
    Escolhe o primeiro resultado aceitável: na cidade certa, com o nome da rua parecido com o da Receita.
    Devolve (lat, lon, sufixo, confiança, rua, nota) ou None, e um motivo quando nada serve.
    """
    if not resposta:
        return None, "sem resultado"
    motivo = "fora da cidade"
    for item in resposta:
        endereco = item.get("address", {})
        cidade = endereco.get("city") or endereco.get("town") or endereco.get("municipality") or endereco.get("village") or ""
        if normalizar_texto(pd.Series([cidade])).iloc[0] != CIDADE_NORMALIZADA:
            continue
        rua = endereco.get("road") or ""
        nota = fuzz.ratio(nucleo_de_texto(rua) or "", nucleo or "")
        if nota < GEOCODING_NOTA_MINIMA_VIA:
            motivo = "rua diferente da Receita"
            continue
        numero_achado = normalizar_numero(pd.Series([endereco.get("house_number", "")])).iloc[0]
        lat, lon = float(item["lat"]), float(item["lon"])
        if isinstance(num, str) and numero_achado == num:
            return (lat, lon, "número encontrado", "baixa", rua, nota), None
        categoria = item.get("category", item.get("class"))
        if categoria == "highway" or item.get("addresstype") == "road":
            return (lat, lon, "só a rua (sem o número)", "muito baixa", rua, nota), None
        motivo = "precisão insuficiente"
    return None, motivo


avaliacoes = {}
motivos_recusa = {}
for chave, num, nucleo in zip(consultas["chave"], consultas["num"], consultas["nucleo"]):
    if chave not in respostas:
        continue
    aceito, motivo = avaliar_resposta(respostas[chave], num, nucleo)
    if aceito:
        avaliacoes[chave] = aceito
    else:
        motivos_recusa[motivo] = motivos_recusa.get(motivo, 0) + 1
log.info(f"Etapa 4 concluída! Endereços aceitos: {len(avaliacoes):,}.")
for nome_motivo, qtd in motivos_recusa.items():
    log.info(f"    Recusados por {nome_motivo}: {qtd:,}")

# ===================================================================
# ETAPA 5: GRAVAR E RESUMIR
# ===================================================================
log.info("\nEtapa 5: gravando o resultado no df_espacial...")
aceitos = est[est["chave"].isin(avaliacoes)]
if len(aceitos):
    valores = [avaliacoes[c] for c in aceitos["chave"]]
    resultado = pd.DataFrame(valores, columns=["lat", "lon", "sufixo", "confianca", "via_candidata", "nota_fuzzy"],
                             index=aceitos["idx"].values)
    resultado["metodo"] = "OpenStreetMap: " + NOME_PASSO + " — " + resultado["sufixo"]
    resultado["base"] = "OpenStreetMap (Nominatim)"
    resultado = resultado.drop(columns=["sufixo"])
else:
    resultado = pd.DataFrame()
gravar_resultado(df_espacial, resultado, 13)
log.info("Etapa 5 concluída!")
resumo_passo(df_espacial, 13, NOME_PASSO, len(est), resultado)
log.info("\nAtribuição exigida: dados de geocoding © colaboradores do OpenStreetMap (licença ODbL).")

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 2. Gravação e validação
Grava o `df_espacial` após o passo 13.

In [ ]:
log.info("SEÇÃO 2: gravação e validação...")
validacao.validar_tabela("df_espacial", df_espacial, chave="cnpj", obrigatorias=["cnpj"], linhas_entrada=len(aux_estab))
fontes.gravar_processamento(cfg, "espacial_06", df_espacial)
log.info("SEÇÃO 2 concluída!")

## 3. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 06_geocoding_nominatim...")
com_ponto = int(df_espacial["lat"].notna().sum())
log.info(f"Com coordenada após o passo 13: {com_ponto:,} de {len(df_espacial):,} ({com_ponto / len(df_espacial) * 100:.1f}%)")
log.info("Arquivos gerados:")
for chave in ['espacial_06']:
    caminho = cfg.caminho("processamento." + chave)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")